# Scalability & Convergence Test

Validates the **NumPy-vectorized** Monte Carlo simulation across increasing workloads and alt_weight values.

## Methods Tested (6)

| Method | Description |
| --- | --- |
| `historical` | Random sampling from historical log returns |
| `window` | Consecutive T-day window from random start |
| `window_10d` | Window with 10-day step |
| `window_20d` | Window with 20-day step |
| `student_t` | (1-alt_weight) historical + alt_weight Student-t draws |
| `black_scholes` | GBM under risk-neutral measure (benchmark) — uses live r from ^IRX and annualized vol |

## Test Dimensions

- **Scales:** 1K → 10M paths (8 levels)
- **Alt weights:** 0.1, 0.2, 0.3, 0.4, 0.5
- **Convergence threshold:** <1% price change between consecutive scales

## Output

Timing table, convergence report, pass/fail per method, convergence charts

In [0]:
%pip install scipy yfinance --quiet
dbutils.library.restartPython()

In [0]:
import sys
import os
import time

import pandas as pd

src_dir = "/Workspace/Users/spyros.louvis@ctpsandbox.com/pub-python-repo/databricks/src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from config.settings import FULL_TABLE_NAME, get_logger
from utils.simulation_helpers import (
    load_historical_data, fit_distributions, run_simulations, get_risk_free_rate,
)

logger = get_logger("scalability_test")
logger.info("Modules loaded.")

In [0]:
# ---------------------------------------------------------------------------
# Test configuration
# ---------------------------------------------------------------------------
TICKER = "^GSPC"
STRIKE_K = 6500.0
PERIOD_T = 10

# Scales to test (NumPy can handle much larger scales than UDFs)
RUN_SCALES = [1_000, 10_000, 100_000, 500_000, 1_000_000, 2_000_000, 5_000_000, 10_000_000]

# Convergence threshold: max allowed % change between consecutive scales
CONVERGENCE_THRESHOLD_PCT = 1.0

# Alternative distribution weights to test
ALT_WEIGHTS = [0.1, 0.2, 0.3, 0.4, 0.5]

logger.info(f"Config: ticker={TICKER}, K={STRIKE_K}, T={PERIOD_T}")
logger.info(f"Scales: {RUN_SCALES}")
logger.info(f"Alt weights to test: {ALT_WEIGHTS}")
logger.info(f"Convergence threshold: {CONVERGENCE_THRESHOLD_PCT}%")

In [0]:
# ---------------------------------------------------------------------------
# Load data, fit distributions, and fetch risk-free rate
# ---------------------------------------------------------------------------
log_returns, S0 = load_historical_data(spark, FULL_TABLE_NAME, TICKER)
dist_params = fit_distributions(log_returns)
r = get_risk_free_rate()

logger.info(f"S0 = {S0:.2f}, vol = {dist_params['vol']:.4f}, r = {r:.4f}")
logger.info(f"{len(log_returns):,} log returns loaded.")

In [0]:
# ---------------------------------------------------------------------------
# Run simulations at each scale x alt_weight (NumPy vectorized)
# ---------------------------------------------------------------------------
timing_records = []
price_records = []

for alt_w in ALT_WEIGHTS:
    logger.info(f"\n{'#'*60}")
    logger.info(f"ALT_WEIGHT = {alt_w}")
    logger.info(f"{'#'*60}")

    for num_runs in RUN_SCALES:
        t_start = time.time()

        results_pdf = run_simulations(
            log_returns, S0, STRIKE_K, PERIOD_T, num_runs,
            dist_params=dist_params,
            alt_weight=alt_w,
            r=r,
        )

        elapsed = time.time() - t_start
        timing_records.append({
            "alt_weight": alt_w,
            "runs": num_runs,
            "elapsed_sec": round(elapsed, 4),
        })
        logger.info(f"  {num_runs:>10,} runs | {elapsed:.4f}s (alt_weight={alt_w})")

        # Store prices per method
        for _, row in results_pdf.iterrows():
            price_records.append({
                "alt_weight": alt_w,
                "runs": num_runs,
                "method": row["method"],
                "CallPrice": row["CallPrice"],
                "PutPrice": row["PutPrice"],
            })

logger.info("\nAll weight x scale combinations completed.")

In [0]:
# ---------------------------------------------------------------------------
# Performance results
# ---------------------------------------------------------------------------
timing_df = pd.DataFrame(timing_records)
timing_df["runs_formatted"] = timing_df["runs"].apply(lambda x: f"{x:,}")

print("\n" + "=" * 60)
print("PERFORMANCE RESULTS")
print("=" * 60)
display(spark.createDataFrame(
    timing_df[["alt_weight", "runs_formatted", "elapsed_sec"]]
))

In [0]:
# ---------------------------------------------------------------------------
# Convergence analysis: check if prices change < 1% between scale jumps
# (grouped by alt_weight and method)
# ---------------------------------------------------------------------------
price_df = pd.DataFrame(price_records)

convergence_records = []

for alt_w in ALT_WEIGHTS:
    for method in price_df["method"].unique():
        mask = (price_df["alt_weight"] == alt_w) & (price_df["method"] == method)
        method_df = price_df[mask].sort_values("runs").reset_index(drop=True)

        for i in range(1, len(method_df)):
            prev = method_df.iloc[i - 1]
            curr = method_df.iloc[i]

            # Use whichever price is non-zero (Call or Put)
            prev_price = prev["CallPrice"] if prev["CallPrice"] > 0 else prev["PutPrice"]
            curr_price = curr["CallPrice"] if curr["CallPrice"] > 0 else curr["PutPrice"]

            if prev_price == 0 and curr_price == 0:
                pct_change = 0.0
            elif prev_price == 0:
                pct_change = 100.0
            else:
                pct_change = abs((curr_price - prev_price) / prev_price) * 100

            converged = pct_change <= CONVERGENCE_THRESHOLD_PCT

            convergence_records.append({
                "alt_weight": alt_w,
                "method": method,
                "from_runs": int(prev["runs"]),
                "to_runs": int(curr["runs"]),
                "prev_price": round(prev_price, 4),
                "curr_price": round(curr_price, 4),
                "pct_change": round(pct_change, 4),
                "converged": converged,
            })

convergence_df = pd.DataFrame(convergence_records)

print("\n" + "=" * 60)
print(f"CONVERGENCE ANALYSIS (threshold: {CONVERGENCE_THRESHOLD_PCT}%)")
print("=" * 60)
display(spark.createDataFrame(convergence_df))

In [0]:
# ---------------------------------------------------------------------------
# Price convergence curves (faceted by alt_weight)
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

n_weights = len(ALT_WEIGHTS)
fig, axes = plt.subplots(n_weights, 2, figsize=(14, 4 * n_weights), squeeze=False)

methods = price_df["method"].unique()
colors = plt.cm.tab10(range(len(methods)))

for row_idx, alt_w in enumerate(ALT_WEIGHTS):
    ax1 = axes[row_idx, 0]
    ax2 = axes[row_idx, 1]

    # Left: absolute prices
    for i, method in enumerate(methods):
        mask = (price_df["alt_weight"] == alt_w) & (price_df["method"] == method)
        mdf = price_df[mask].sort_values("runs")
        prices = mdf.apply(lambda r: r["CallPrice"] if r["CallPrice"] > 0 else r["PutPrice"], axis=1)
        ax1.plot(mdf["runs"], prices, marker="o", label=method, color=colors[i], linewidth=1.5, markersize=4)

    ax1.set_xscale("log")
    ax1.set_xlabel("Simulations (log)")
    ax1.set_ylabel("Option Price ($)")
    ax1.set_title(f"Price Convergence (alt_weight={alt_w})")
    ax1.legend(loc="best", fontsize=7)
    ax1.grid(True, alpha=0.3)
    ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

    # Right: % change (convergence rate)
    conv_w = convergence_df[convergence_df["alt_weight"] == alt_w]
    for i, method in enumerate(methods):
        cdf = conv_w[conv_w["method"] == method].sort_values("to_runs")
        if len(cdf) > 0:
            ax2.plot(range(len(cdf)), cdf["pct_change"], marker="s", label=method,
                     color=colors[i], linewidth=1.5, markersize=4)

    ax2.axhline(y=CONVERGENCE_THRESHOLD_PCT, color="red", linestyle="--", linewidth=1.5,
                label=f"Threshold ({CONVERGENCE_THRESHOLD_PCT}%)")

    # X-axis labels
    sample_cdf = conv_w[conv_w["method"] == methods[0]].sort_values("to_runs")
    tick_labels = [f"{int(r['from_runs']):,}\u2192{int(r['to_runs']):,}" for _, r in sample_cdf.iterrows()]
    ax2.set_xticks(range(len(tick_labels)))
    ax2.set_xticklabels(tick_labels, fontsize=7, rotation=15)
    ax2.set_xlabel("Scale Transition")
    ax2.set_ylabel("% Change")
    ax2.set_title(f"Convergence Rate (alt_weight={alt_w})")
    ax2.legend(loc="best", fontsize=7)
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/convergence_curves_weights.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved to /tmp/convergence_curves_weights.png")

In [0]:
# ---------------------------------------------------------------------------
# Summary: pass/fail report per alt_weight
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("CONVERGENCE SUMMARY BY ALT_WEIGHT")
print("=" * 60)

# Only check convergence at the highest scale transition
high_from = RUN_SCALES[-2]
high_to = RUN_SCALES[-1]

weight_summary = []

for alt_w in ALT_WEIGHTS:
    high_scale = convergence_df[
        (convergence_df["alt_weight"] == alt_w) &
        (convergence_df["from_runs"] == high_from)
    ]

    print(f"\n  alt_weight = {alt_w}:")
    n_passed = 0
    n_total = len(high_scale)
    for _, row in high_scale.iterrows():
        status = "PASS" if row["converged"] else "FAIL"
        if row["converged"]:
            n_passed += 1
        print(f"    [{status}] {row['method']:12s} | {row['pct_change']:.2f}% change")

    weight_summary.append({
        "alt_weight": alt_w,
        "methods_converged": n_passed,
        "total_methods": n_total,
        "best_method": high_scale.loc[high_scale["pct_change"].idxmin(), "method"] if n_total > 0 else "N/A",
        "best_pct_change": high_scale["pct_change"].min() if n_total > 0 else None,
    })

print(f"\n{'='*60}")
print("WEIGHT COMPARISON SUMMARY")
print(f"{'='*60}")
print(f"  Scale transition: {high_from:,} \u2192 {high_to:,} runs")
print(f"  Threshold: {CONVERGENCE_THRESHOLD_PCT}%")
print()

all_pass = True
for ws in weight_summary:
    verdict = "\u2705" if ws['methods_converged'] == ws['total_methods'] else "\u26a0\ufe0f"
    if ws['methods_converged'] < ws['total_methods']:
        all_pass = False
    print(f"  {verdict} alt_weight={ws['alt_weight']:.1f}: "
          f"{ws['methods_converged']}/{ws['total_methods']} converged | "
          f"best: {ws['best_method']} ({ws['best_pct_change']:.2f}%)")

print(f"\n  Overall: {'ALL WEIGHTS PASS \u2705' if all_pass else 'SOME WEIGHTS HAVE FAILURES'}")

# Performance summary
print(f"\n{'='*60}")
print("PERFORMANCE SUMMARY (avg time per scale across weights)")
print(f"{'='*60}")
for scale in RUN_SCALES:
    avg_time = timing_df[timing_df["runs"] == scale]["elapsed_sec"].mean()
    print(f"  {scale:>10,} runs \u2192 {avg_time:>7.2f}s (avg)")

In [0]:
# ---------------------------------------------------------------------------
# Compare Black-Scholes prices against empirical methods at highest scale
# ---------------------------------------------------------------------------
import numpy as np

high_scale = RUN_SCALES[-1]
high_df = price_df[price_df["runs"] == high_scale].copy()

print(f"{'='*75}")
print(f"PRICE COMPARISON AT {high_scale:,} RUNS (T={PERIOD_T}, K={STRIKE_K}, S0={S0:.2f})")
print(f"{'='*75}")

# Black-Scholes prices (should be constant across alt_weights)
bs_prices = high_df[high_df["method"] == "black_scholes"][["alt_weight", "CallPrice", "PutPrice"]]
bs_call_mean = bs_prices["CallPrice"].mean()
bs_put_mean = bs_prices["PutPrice"].mean()
bs_call_std = bs_prices["CallPrice"].std()
bs_put_std = bs_prices["PutPrice"].std()

print(f"\n  Black-Scholes benchmark (r={r:.4f}, vol={dist_params['vol']:.4f}):")
print(f"    Call: ${bs_call_mean:.4f} (std across weights: ${bs_call_std:.4f})")
print(f"    Put:  ${bs_put_mean:.4f} (std across weights: ${bs_put_std:.4f})")
print(f"    {'\u2705 Consistent' if bs_call_std < 1.0 else '\u26a0\ufe0f Inconsistent'} across alt_weights (as expected)")

# Compare all methods at each alt_weight
print(f"\n{'\u2500'*75}")
print(f"  {'Method':<14s} {'alt_w=0.1':>10s} {'alt_w=0.2':>10s} {'alt_w=0.3':>10s} {'alt_w=0.4':>10s} {'alt_w=0.5':>10s} {'vs BS':>8s}")
print(f"{'\u2500'*75}")

for method in price_df["method"].unique():
    row_data = []
    for alt_w in ALT_WEIGHTS:
        mask = (high_df["alt_weight"] == alt_w) & (high_df["method"] == method)
        call = high_df[mask]["CallPrice"].values[0]
        row_data.append(call)
    
    avg_price = np.mean(row_data)
    diff_vs_bs = ((avg_price - bs_call_mean) / bs_call_mean * 100) if bs_call_mean > 0 else 0
    
    prices_str = "  ".join(f"${v:>8.2f}" for v in row_data)
    print(f"  {method:<14s} {prices_str}  {diff_vs_bs:>+6.1f}%")

# Summary: BS vs empirical spread
print(f"\n{'='*75}")
print(f"INTERPRETATION")
print(f"{'='*75}")
print(f"  - Black-Scholes uses risk-neutral pricing (drift=r, discounted to PV)")
print(f"  - Empirical methods use real-world measure (historical drift, no discount)")
print(f"  - Difference reflects risk premium: empirical > BS when market drift > r")
print(f"  - student_t is alt_weight-sensitive; black_scholes is alt_weight-invariant")

In [0]:
# ---------------------------------------------------------------------------
# Bar chart: Call prices by method at highest scale
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

high_scale = RUN_SCALES[-1]
high_df = price_df[price_df["runs"] == high_scale].copy()
methods_list = list(price_df["method"].unique())
n_methods = len(methods_list)
n_weights = len(ALT_WEIGHTS)

# Analytical Black-Scholes price
vol = dist_params["vol"]
T_years = PERIOD_T / 252.0
d1 = (np.log(S0 / STRIKE_K) + (r + 0.5 * vol**2) * T_years) / (vol * np.sqrt(T_years))
d2 = d1 - vol * np.sqrt(T_years)
bs_analytical_call = S0 * norm.cdf(d1) - STRIKE_K * np.exp(-r * T_years) * norm.cdf(d2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: Grouped bar chart (Call prices by method, grouped by alt_weight) ---
bar_width = 0.14
x = np.arange(n_methods)
colors = plt.cm.viridis(np.linspace(0.2, 0.9, n_weights))

for i, alt_w in enumerate(ALT_WEIGHTS):
    calls = []
    for method in methods_list:
        mask = (high_df["alt_weight"] == alt_w) & (high_df["method"] == method)
        calls.append(high_df[mask]["CallPrice"].values[0])
    offset = (i - n_weights / 2 + 0.5) * bar_width
    ax1.bar(x + offset, calls, bar_width, label=f"w={alt_w}", color=colors[i])

# MC BS mean reference line
bs_mc_mean = high_df[high_df["method"] == "black_scholes"]["CallPrice"].mean()
ax1.axhline(y=bs_mc_mean, color="red", linestyle="--", linewidth=1.5, label=f"BS MC (${bs_mc_mean:.0f})")

# Analytical BS reference line
ax1.axhline(y=bs_analytical_call, color="green", linestyle="-.", linewidth=1.5, label=f"BS Analytical (${bs_analytical_call:.0f})")

ax1.set_xticks(x)
ax1.set_xticklabels(methods_list, rotation=20, ha="right", fontsize=9)
ax1.set_ylabel("Call Price ($)")
ax1.set_title(f"Call Prices by Method at {high_scale:,} Runs (T={PERIOD_T})")
ax1.legend(fontsize=8, loc="lower left")
ax1.grid(axis="y", alpha=0.3)
ax1.set_ylim(bottom=700)

# --- Right: Premium over BS analytical (%) ---
method_premiums = []
for method in methods_list:
    method_calls = []
    for alt_w in ALT_WEIGHTS:
        mask = (high_df["alt_weight"] == alt_w) & (high_df["method"] == method)
        method_calls.append(high_df[mask]["CallPrice"].values[0])
    avg_call = np.mean(method_calls)
    premium = (avg_call - bs_analytical_call) / bs_analytical_call * 100
    method_premiums.append(premium)

bar_colors = ["#2ecc71" if p > 0 else "#e74c3c" for p in method_premiums]
ax2.barh(methods_list, method_premiums, color=bar_colors, edgecolor="white", linewidth=0.5)
ax2.axvline(x=0, color="black", linewidth=0.8)
ax2.set_xlabel("Premium over BS Analytical (%)")
ax2.set_title(f"Avg Call Premium vs BS Analytical (${bs_analytical_call:.2f})")
ax2.grid(axis="x", alpha=0.3)

for i, (method, prem) in enumerate(zip(methods_list, method_premiums)):
    ax2.text(prem + 0.05, i, f"{prem:+.1f}%", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("/tmp/methods_vs_bs_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"BS Analytical Call: ${bs_analytical_call:.4f} | BS MC Mean: ${bs_mc_mean:.4f} | Diff: ${bs_mc_mean - bs_analytical_call:.4f}")